# Sales Forecasting using Adaptive Basis Function Models
## Complete Machine Learning & Econometric Time Series Project (Steps 1 – 9)

---
### Executive Summary & Problem Formulation
Accurate sales forecasting in multi-echelon retail environments is critical for optimizing working capital, minimizing stockouts, and scheduling staffing. Retail demand time series are characterized by:
1. **Strong deterministic and stochastic seasonality** (52-week annual cycles, Black Friday / Thanksgiving surges, Christmas peaks).
2. **Exogenous macroeconomic influences** (fuel prices, CPI, unemployment, promotional markdowns).
3. **Non-linear interactions** across store types, floor space, and calendar events.

Traditional regression assumes fixed linear relationships $\hat{y} = \mathbf{w}^T \mathbf{x}$. Fixed basis expansions (Polynomial, Fourier) introduce non-linearities but use rigid, data-independent transformations $\phi_j(\mathbf{x})$.
This project applies **Adaptive Basis Function Models** where the basis functions $\phi_j(\mathbf{x}; \boldsymbol{\theta}_j)$ are parameterized and adaptively optimized directly from data:
$$\hat{y}(\mathbf{x}) = w_0 + \sum_{j=1}^M w_j \phi_j(\mathbf{x}; \boldsymbol{\theta}_j)$$
where both linear weights $w_j$ and basis parameters $\boldsymbol{\theta}_j$ (such as hidden layer weights, radial centers, spline knots, or decision tree splits) are learned.

---
### Dataset & Scope
* **Dataset**: Kaggle *Walmart Recruiting - Store Sales Forecasting* (`train.csv`, `features.csv`, `stores.csv`).
* **Timeline**: February 2010 to October 2012 (143 weekly timestamps across 45 stores).
* **Primary Forecasting Scope**: Total weekly sales per store (aggregated across departments) evaluated on a strict 16-week chronological holdout test set with 5-fold expanding-window `TimeSeriesSplit` cross-validation.


## 0. Environment Setup & Reproducibility
We fix the global random seed to `42` across Python, NumPy, Scikit-Learn, and gradient boosting frameworks. Dynamic path resolution enables execution across Google Colab, JupyterLab, and command-line environments.

In [1]:
# Environment Configuration & Path Resolution
import os
import sys
import time
import random
import warnings
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf

# Set random seed
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# Formatting
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'Arial'

# Dynamic path resolution
CURRENT_PATH = pathlib.Path('.').resolve()
PROJECT_ROOT = CURRENT_PATH.parent if CURRENT_PATH.name == 'notebooks' else CURRENT_PATH

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
FIGURES_DIR = os.path.join(RESULTS_DIR, 'figures')
TABLES_DIR = os.path.join(RESULTS_DIR, 'tables')
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')

for d in [DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR, SRC_DIR]:
    os.makedirs(d, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Random seed: {SEED}")


Project root: D:\pbl-4
Random seed: 42


## Step 1: Data Understanding & Exploratory Data Analysis (EDA)

We ingest the three raw CSV files, inspect schemas, handle missing promotional markdowns, and aggregate sales to the store level.

In [2]:
# Ingest Raw Datasets & Verify Panel Balance
stores = pd.read_csv(os.path.join(DATA_DIR, 'stores.csv'))
features = pd.read_csv(os.path.join(DATA_DIR, 'features.csv'))
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))

features['Date'] = pd.to_datetime(features['Date'])
train['Date'] = pd.to_datetime(train['Date'])

print(f"stores.csv:   {stores.shape}   | Types: {stores['Type'].value_counts().to_dict()}")
print(f"features.csv: {features.shape} | Date range: {features['Date'].min().date()} to {features['Date'].max().date()}")
print(f"train.csv:    {train.shape} | Date range: {train['Date'].min().date()} to {train['Date'].max().date()}")

# Aggregate sales across departments to store-level
store_weekly = train.groupby(['Store', 'Date', 'IsHoliday'], as_index=False)['Weekly_Sales'].sum()

# Merge master balanced panel
merged = store_weekly.merge(stores, on='Store', how='left')
merged = merged.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
merged = merged.sort_values(by=['Store', 'Date']).reset_index(drop=True)

# Impute MarkDown NaNs with 0.0 and create indicator
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
merged['markdown_available'] = (
    merged[markdown_cols].notnull().any(axis=1) & 
    (merged[markdown_cols].fillna(0) > 0).any(axis=1)
).astype(int)

for col in markdown_cols:
    merged[col] = merged[col].fillna(0.0)

merged['IsHoliday'] = merged['IsHoliday'].astype(int)

print(f"Master merged store panel: {merged.shape} (Expected: 45 stores * 143 weeks = 6,435 rows)")


stores.csv:   (45, 3)   | Types: {'A': 22, 'B': 17, 'C': 6}
features.csv: (8190, 12) | Date range: 2010-02-05 to 2013-07-26
train.csv:    (421570, 5) | Date range: 2010-02-05 to 2012-10-26
Master merged store panel: (6435, 16) (Expected: 45 stores * 143 weeks = 6,435 rows)


### 1.2 Diagnostic Time Series Properties: STL Decomposition & ADF Stationarity
We run Seasonal-Trend decomposition using LOESS (STL) with an annual period $P=52$ and execute Augmented Dickey-Fuller (ADF) unit-root tests.

In [3]:
# STL Decomposition & ADF Stationarity Testing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

chain_weekly = merged.groupby('Date').agg({'Weekly_Sales': 'sum', 'IsHoliday': 'max'}).reset_index()
chain_ts = chain_weekly.set_index('Date')['Weekly_Sales'].asfreq('W-FRI')

stl_res = STL(chain_ts, period=52, robust=True).fit()

# Augmented Dickey-Fuller Tests
adf_raw = adfuller(chain_ts.dropna(), autolag='AIC')
adf_diff = adfuller(chain_ts.diff().dropna(), autolag='AIC')

adf_df = pd.DataFrame({
    'Series': ['Raw Total Sales', 'First Difference (Δ Sales)'],
    'ADF_Statistic': [adf_raw[0], adf_diff[0]],
    'p_value': [adf_raw[1], adf_diff[1]],
    'Critical_Value_5%': [adf_raw[4]['5%'], adf_diff[4]['5%']],
    'Stationary (p < 0.05)': [adf_raw[1] < 0.05, adf_diff[1] < 0.05]
})
display(adf_df)


,Series,ADF_Statistic,p_value,Critical_Value_5%,Stationary (p < 0.05)
0,Raw Total Sales,-5.908,0.000,-2.883,True
1,First Difference (Δ Sales),-6.699,0.000,-2.883,True


## Step 2: Zero-Leakage Feature Engineering
We generate 37 features across Calendar/Cyclical, Holiday Proximity, Autoregressive Lags, Shifted Rolling Windows, Store Attributes, and Macroeconomic Indicators. All rolling statistics are strictly calculated with `.shift(1)` to eliminate lookahead bias.

In [4]:
# Run Feature Pipeline
from src.features import prepare_feature_pipeline, get_feature_columns, time_based_train_test_split, scale_features

df_features = prepare_feature_pipeline(merged, lags=[1, 2, 4, 52], rolling_windows=[4, 12], drop_na_lags=True)
predictor_cols = get_feature_columns(df_features)

print(f"Features engineered shape: {df_features.shape}")
print(f"Predictor feature count:  {len(predictor_cols)}")


Features engineered shape: (4095, 42)
Predictor feature count:  38


## Step 3: Train / Test Split & TimeSeries Validation Strategy
We partition the dataset chronologically:
* **Training Partition**: February 2011 to July 6, 2012 (75 weeks $\times$ 45 stores = 3,375 rows).
* **Holdout Testing Partition**: July 13, 2012 to October 26, 2012 (16 weeks $\times$ 45 stores = 720 rows).
* `StandardScaler` is fitted **exclusively on the training partition**.

In [5]:
# Chronological Split and Scaling
train_df, test_df, split_date = time_based_train_test_split(df_features, test_weeks=16)
train_scaled, test_scaled, scaler = scale_features(train_df, test_df, predictor_cols, scaler_type='standard')

X_train = train_scaled[predictor_cols]
y_train = train_scaled['Weekly_Sales'].values
X_test = test_scaled[predictor_cols]
y_test = test_scaled['Weekly_Sales'].values
is_holiday_test = test_df['IsHoliday'].values

print(f"Train split: {train_df['Date'].min().date()} to {train_df['Date'].max().date()} ({X_train.shape})")
print(f"Test split:  {test_df['Date'].min().date()} to {test_df['Date'].max().date()} ({X_test.shape})")


Train split: 2011-02-04 to 2012-07-06 ((3375, 38))
Test split:  2012-07-13 to 2012-10-26 ((720, 38))


## Step 4: Model Architectures & Mathematical Foundations

Every model is conceptualized under the general basis function framework:
$$\hat{y}(\mathbf{x}) = w_0 + \sum_{j=1}^M w_j \phi_j(\mathbf{x}; \boldsymbol{\theta}_j)$$

### Model Categorization:
1. **Baselines**:
   * **Naive Forecaster**: $\phi_1(\mathbf{x}) = y_{t-1}, w_1 = 1$.
   * **Seasonal Naive**: $\phi_1(\mathbf{x}) = y_{t-52}, w_1 = 1$.
   * **Linear & Ridge Regression**: $\phi_j(\mathbf{x}) = x_j$ with $\ell_2$ shrinkage.
2. **Fixed Basis Function Models**:
   * **Polynomial Regression**: $\phi_j(\mathbf{x}) = x_i^d$ (degree $d=2$). Bases are data-independent.
   * **Fourier Feature Regression**: $\phi_k(t) = [\sin(2\pi k t / P), \cos(2\pi k t / P)]$.
3. **Adaptive Basis Function Models**:
   * **Multi-Layer Perceptron (MLP)**: $\phi_j(\mathbf{x}) = \sigma(\mathbf{w}_j^T \mathbf{x} + b_j)$. Bases are learned via backpropagation.
   * **Radial Basis Function (RBF) Network**: $\phi_j(\mathbf{x}) = \exp(-\gamma \|\mathbf{x} - \boldsymbol{\mu}_j\|^2)$. Centers $\boldsymbol{\mu}_j$ are learned adaptively via $K$-Means clustering.
   * **Multivariate Adaptive Regression Splines (MARS)**: $\phi_j(\mathbf{x}) = [\pm(x_i - t)]_+$. Piecewise linear/cubic splines with adaptive quantile knots.
   * **Generalized Additive Model (GAM)**: $\hat{y} = \beta_0 + \sum f_j(x_j)$ with penalized B-spline bases.
   * **Decision Trees & Random Forests**: $\phi_m(\mathbf{x}) = \mathbb{I}(\mathbf{x} \in R_m)$. Adaptive recursive partitioning of the feature space.
   * **Gradient Boosted Trees (GBDT, XGBoost, LightGBM)**: Adaptively adds weak learners via functional gradient descent.

## Step 5: Hyperparameter Tuning via TimeSeriesSplit
We tune tree depth, learning rates, regularization alphas, and hidden layer dimensions using a 3-fold expanding-window `TimeSeriesSplit` on the training partition.

In [6]:
# Load Saved Tuning Results
tuning_df = pd.read_csv(os.path.join(TABLES_DIR, 'hyperparameter_tuning.csv'))
display(tuning_df)


,Model,Tuned_Parameters,Best_CV_Score (MAE)
0,MLP Neural Network,"{'hidden_layer_sizes': (128, 64), 'alpha': 0.01}",717717.834
1,Decision Tree,"{'min_samples_leaf': 10, 'max_depth': 12}",68646.845
2,Random Forest,"{'n_estimators': 100, 'min_samples_leaf': 3, '...",56730.685
3,Gradient Boosting,"{'n_estimators': 120, 'max_depth': 4, 'learnin...",53791.255
4,XGBoost,"{'n_estimators': 150, 'max_depth': 5, 'learnin...",53224.237
5,LightGBM,"{'n_estimators': 150, 'max_depth': 6, 'learnin...",55729.326


## Step 6: Evaluation Metrics on the Holdout Test Set
We evaluate all 15 model variations across MAE, RMSE, MAPE, sMAPE, $R^2$, MASE, and Weighted MAE (WMAE, with $5\times$ weight on holiday weeks):
$$\text{WMAE} = \frac{\sum_{i=1}^N w_i |y_i - \hat{y}_i|}{\sum_{i=1}^N w_i}, \quad w_i = \begin{cases} 5 & \text{if Holiday} \\ 1 & \text{otherwise} \end{cases}$$

In [7]:
# Display Model Comparison Metrics Table
metrics_df = pd.read_csv(os.path.join(TABLES_DIR, 'model_comparison_metrics.csv'))
display(metrics_df[['Model', 'WMAE', 'MAE', 'RMSE', 'MAPE (%)', 'R2', 'MASE', 'Holiday_MAE', 'NonHoliday_MAE']])


,Model,WMAE,MAE,RMSE,MAPE (%),R2,MASE,Holiday_MAE,NonHoliday_MAE
0,Random Forest,40872.592,39928.987,64169.208,3.852,0.985,0.773,44647.009,39614.452
1,XGBoost,42455.823,42167.967,66945.885,4.115,0.984,0.817,43607.250,42072.014
2,Gradient Boosting,42674.073,41970.389,66167.756,4.090,0.984,0.813,45488.808,41735.827
3,LightGBM,45129.099,43998.954,67850.909,4.215,0.983,0.852,49649.680,43622.239
4,Ridge Regression,45240.022,44414.232,60799.833,4.729,0.987,0.860,48543.182,44138.968
5,Linear Regression,47558.421,46435.070,64783.644,5.079,0.985,0.899,52051.826,46060.619
6,Decision Tree,47968.883,48051.602,74225.000,4.683,0.980,0.931,47638.010,48079.175
7,MARS (SplineTransformer),48576.210,47989.487,67196.532,5.266,0.984,0.929,50923.104,47793.912
8,Polynomial Regression (deg 2),51644.329,49050.119,68341.975,5.011,0.983,0.950,62021.168,48185.382
9,Seasonal Naive (Lag 52),54154.801,51634.906,82827.975,5.346,0.975,1.000,64234.381,50794.941


In [8]:
# Display Diebold-Mariano Hypothesis Test Result
dm_df = pd.read_csv(os.path.join(TABLES_DIR, 'diebold_mariano_test.csv'))
print("=== DIEBOLD-MARIANO TEST (Best Adaptive vs Best Baseline) ===")
display(dm_df)


=== DIEBOLD-MARIANO TEST (Best Adaptive vs Best Baseline) ===


,Baseline_Model,Adaptive_Model,DM_Statistic,p_value,Significantly_Superior_at_5%
0,Ridge Regression,Random Forest,-1.066,0.287,False


## Step 7: Model Diagnostics & Interpretability Analysis
We analyze actual vs predicted trajectories, residual distributions, feature attributions via SHAP, GAM spline partial dependencies, RBF learned centers, learning curves, and feature ablation impacts.

In [9]:
# Display Ablation Study Results
ablation_df = pd.read_csv(os.path.join(TABLES_DIR, 'ablation_study_results.csv'))
print("=== FEATURE ABLATION STUDY ===")
display(ablation_df)


=== FEATURE ABLATION STUDY ===


,Configuration,Num_Features,WMAE,RMSE,R2,WMAE_Increase (%)
0,Full Model (All Features),38,44254.182,66544.285,0.984,0.000
1,Without Lag Features,30,82314.903,118569.359,0.949,86.005
2,Without Holiday Features,31,43003.607,65274.073,0.985,-2.826
3,Without Macro / External,27,46665.563,69987.796,0.982,5.449


In [10]:
# Display Forecast Error by Store Type
type_error_df = pd.read_csv(os.path.join(TABLES_DIR, 'error_by_store_type.csv'))
print("=== FORECAST ERROR BY STORE TYPE ===")
display(type_error_df)


=== FORECAST ERROR BY STORE TYPE ===


,Store_Type,Store_Count,Mean_Weekly_Sales,MAE,RMSE,WMAE,MAPE (%)
0,A,22,1348262.264,53998.476,79532.211,53450.340,4.118
1,B,17,798504.089,31441.573,51220.950,34313.690,3.964
2,C,6,482635.332,12388.534,16004.017,13337.734,2.557


## Step 8: Multi-Step Ahead Forecasting & Managerial Strategy
The champion model is retrained on all 91 weeks of panel data to generate a **12-week multi-step forward forecast** (November 2012 to January 2013), complete with $80\%$ and $95\%$ prediction intervals derived via empirical residual bootstrap.

In [11]:
# Display 12-Week Forward Forecast Summary
forecast_df = pd.read_csv(os.path.join(TABLES_DIR, 'twelve_week_forecast.csv'))
display(forecast_df)


,Date,Forecast_Sales,Lower_95,Upper_95,Lower_80,Upper_80,IsHoliday
0,2012-11-02,48549589.634,45118338.353,51980840.914,46308772.471,50790406.797,0
1,2012-11-09,48828912.605,45397661.325,52260163.885,46588095.442,51069729.768,0
2,2012-11-16,47689708.820,44258457.540,51120960.101,45448891.657,49930525.983,0
3,2012-11-23,61532374.583,58101123.303,64963625.863,59291557.420,63773191.746,1
4,2012-11-30,48368547.847,44937296.567,51799799.128,46127730.684,50609365.010,0
5,2012-12-07,52888679.277,49457427.996,56319930.557,50647862.114,55129496.439,0
6,2012-12-14,56894388.923,53463137.642,60325640.203,54653571.760,59135206.085,0
7,2012-12-21,70525880.947,67094629.666,73957132.227,68285063.784,72766698.109,1
8,2012-12-28,51618061.317,48186810.037,55049312.598,49377244.155,53858878.480,1
9,2013-01-04,47757609.848,44326358.567,51188861.128,45516792.685,49998427.011,0


## Step 9: Project Artifacts & File Deliverables Audit
All generated publication-grade figures, metric tables, and academic report drafts are verified below.

In [12]:
# Audit Output Artifacts
print("=== GENERATED FIGURES (results/figures/) ===")
for f in sorted(os.listdir(FIGURES_DIR)):
    if f.endswith('.png'):
        size_kb = os.path.getsize(os.path.join(FIGURES_DIR, f)) / 1024
        print(f"  - {f:35s} ({size_kb:,.1f} KB)")

print("\n=== GENERATED METRIC TABLES (results/tables/) ===")
for t in sorted(os.listdir(TABLES_DIR)):
    if t.endswith('.csv'):
        size_b = os.path.getsize(os.path.join(TABLES_DIR, t))
        print(f"  - {t:35s} ({size_b:,} bytes)")


=== GENERATED FIGURES (results/figures/) ===
  - ablation_study.png                  (109.4 KB)
  - actual_vs_predicted_top3.png        (1,232.9 KB)
  - adaptive_bases_visualization.png    (521.3 KB)
  - correlation_heatmap.png             (436.0 KB)
  - error_by_store_type.png             (126.9 KB)
  - feature_importance_best_model.png   (146.3 KB)
  - gam_partial_dependence.png          (674.8 KB)
  - holiday_vs_non_holiday_sales.png    (160.1 KB)
  - learning_curves.png                 (160.5 KB)
  - model_rmse_mape_comparison.png      (400.9 KB)
  - overall_sales_trend.png             (279.9 KB)
  - residual_diagnostics.png            (456.0 KB)
  - sales_by_store_type.png             (189.4 KB)
  - sales_vs_external_factors.png       (2,211.2 KB)
  - seasonality_weekly_monthly.png      (251.2 KB)
  - shap_summary.png                    (252.0 KB)
  - stl_decomposition.png               (401.6 KB)
  - timeseries_cv_folds.png             (107.4 KB)
  - twelve_week_ahead_forecast.pn